In [1]:
import requests, pandas as pd, plotly.express as px, plotly.graph_objects as go, time

print("✅ Bibliotecas carregadas!")

jogos = {
    "Elden Ring": 1245620, "Baldur's Gate 3": 1086940, "Deep Rock Galactic": 548430,
    "Fall Guys": 1097150, "Battlefield 2042": 1517290, "Knockout City": 1583530,
    "Stardew Valley": 413150, "Hollow Knight": 367520, "Cyberpunk 2077": 1091500,
    "Among Us": 945360, "Vampire Survivors": 1794680, "Lethal Company": 1966720,
}
categorias = {
    "Elden Ring": "🚀 Explodiu e sustentou", "Baldur's Gate 3": "🚀 Explodiu e sustentou",
    "Deep Rock Galactic": "🚀 Explodiu e sustentou", "Fall Guys": "💥 Explodiu e despencou",
    "Battlefield 2042": "💥 Explodiu e despencou", "Knockout City": "💥 Explodiu e despencou",
    "Stardew Valley": "🐢 Cresceu devagar", "Hollow Knight": "🐢 Cresceu devagar",
    "Cyberpunk 2077": "🔄 Caiu e se recuperou", "Among Us": "🌊 Febre relâmpago",
    "Vampire Survivors": "🌊 Febre relâmpago", "Lethal Company": "🌊 Febre relâmpago",
}
cores = {
    "🚀 Explodiu e sustentou": "#4CAF50", "💥 Explodiu e despencou": "#F44336",
    "🐢 Cresceu devagar": "#2196F3", "🔄 Caiu e se recuperou": "#FF9800", "🌊 Febre relâmpago": "#9C27B0",
}

def coletar(nome, app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&cc=us&l=english"
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).json()
    if not r[str(app_id)]["success"]: return None
    info = r[str(app_id)]["data"]
    return {
        "nome": nome, "app_id": app_id,
        "genero": ", ".join([g["description"] for g in info.get("genres", [])]),
        "desenvolvedor": ", ".join(info.get("developers", [])),
        "data_lancamento": info.get("release_date", {}).get("date", "N/A"),
        "preco_usd": info.get("price_overview", {}).get("final", 0) / 100,
        "metacritic": info.get("metacritic", {}).get("score", None),
        "total_reviews": info.get("recommendations", {}).get("total", None),
    }

dados = []
for nome, app_id in jogos.items():
    print(f"🔍 {nome}...")
    d = coletar(nome, app_id)
    if d: dados.append(d)
    time.sleep(1)

df = pd.DataFrame(dados)
df["data_lancamento"] = pd.to_datetime(df["data_lancamento"], errors="coerce")
df["ano_lancamento"] = df["data_lancamento"].dt.year
df.loc[df["nome"] == "Stardew Valley", "ano_lancamento"] = 2016
df["categoria"] = df["nome"].map(categorias)
print(f"\n✅ {len(df)} jogos coletados e tabela pronta!")

fig = px.bar(df.dropna(subset=["total_reviews"]).sort_values("total_reviews", ascending=True),
    x="total_reviews", y="nome", color="categoria", orientation="h",
    title="📊 Total de Avaliações por Jogo na Steam",
    labels={"total_reviews": "Total de Avaliações", "nome": "Jogo", "categoria": "Categoria"},
    color_discrete_map=cores, text="total_reviews")
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(height=500, plot_bgcolor="white")
fig.show()

fig2 = px.scatter(df.dropna(subset=["metacritic", "total_reviews"]),
    x="metacritic", y="total_reviews", size="preco_usd", color="categoria", text="nome",
    title="🫧 Qualidade vs Popularidade — o preço importa?",
    labels={"metacritic": "Nota Metacritic", "total_reviews": "Total de Avaliações", "categoria": "Categoria"},
    color_discrete_map=cores, size_max=60)
fig2.update_traces(textposition="top center")
fig2.update_layout(height=550, plot_bgcolor="white")
fig2.add_vline(x=90, line_dash="dash", line_color="gray", annotation_text="nota 90+")
fig2.show()

fig3 = px.scatter(df.dropna(subset=["ano_lancamento", "metacritic", "total_reviews"]),
    x="ano_lancamento", y="metacritic", color="categoria", size="total_reviews", text="nome",
    title="📅 Linha do Tempo — Quando cada jogo lançou e qual foi sua qualidade?",
    labels={"ano_lancamento": "Ano de Lançamento", "metacritic": "Nota Metacritic"},
    color_discrete_map=cores, size_max=50)
fig3.update_traces(textposition="top center")
fig3.update_layout(height=550, plot_bgcolor="white", xaxis=dict(tickmode="linear", dtick=1))
fig3.add_hline(y=90, line_dash="dash", line_color="gray", annotation_text="linha dos 90+")
fig3.show()

colunas = ["metacritic", "total_reviews", "preco_usd"]
df_heat = df.dropna(subset=colunas).copy()
for col in colunas:
    df_heat[col+"_norm"] = (df_heat[col]-df_heat[col].min())/(df_heat[col].max()-df_heat[col].min())
df_heat = df_heat.sort_values("categoria")
fig4 = go.Figure(data=go.Heatmap(
    z=df_heat[["metacritic_norm","total_reviews_norm","preco_usd_norm"]].values.T,
    x=df_heat["nome"].tolist(), y=["Metacritic","Total Reviews","Preço (USD)"],
    text=[[f"{v:.0f}" for v in df_heat["metacritic"]],
          [f"{v/1000:.0f}k" for v in df_heat["total_reviews"]],
          [f"${v:.2f}" for v in df_heat["preco_usd"]]],
    texttemplate="%{text}", textfont={"size":11}, colorscale="RdYlGn", showscale=True))
fig4.update_layout(title="🌡️ Heatmap — Comparação Geral", height=350, xaxis=dict(tickangle=-35))
fig4.show()


fig.write_html("grafico1_reviews.html")
fig2.write_html("grafico2_bolhas.html")
fig3.write_html("grafico3_timeline.html")
fig4.write_html("grafico4_heatmap.html")

print("\n🎉 TUDO PRONTO!")
print("📁 4 HTMLs salvos — veja instruções abaixo para baixar")

✅ Bibliotecas carregadas!
🔍 Elden Ring...
🔍 Baldur's Gate 3...
🔍 Deep Rock Galactic...
🔍 Fall Guys...
🔍 Battlefield 2042...
🔍 Knockout City...
🔍 Stardew Valley...
🔍 Hollow Knight...
🔍 Cyberpunk 2077...
🔍 Among Us...
🔍 Vampire Survivors...
🔍 Lethal Company...

✅ 12 jogos coletados e tabela pronta!



🎉 TUDO PRONTO!
📁 4 HTMLs salvos — veja instruções abaixo para baixar


In [2]:
from google.colab import files

files.download("grafico1_reviews.html")
files.download("grafico2_bolhas.html")
files.download("grafico3_timeline.html")
files.download("grafico4_heatmap.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
!pip install pytrends --quiet
print("✅ pytrends instalado!")

✅ pytrends instalado!


In [5]:
from pytrends.request import TrendReq
import time

pytrends = TrendReq(hl="en-US", tz=360)

grupos = [
    ["Elden Ring", "Baldur's Gate 3", "Cyberpunk 2077", "Stardew Valley", "Among Us"],
    ["Fall Guys", "Hollow Knight", "Vampire Survivors", "Lethal Company", "Deep Rock Galactic"],
]

todos_trends = []

for grupo in grupos:
    print(f"🔍 Buscando trends: {grupo}")

    pytrends.build_payload(grupo, timeframe="today 5-y", geo="")

    dados = pytrends.interest_over_time()

    if not dados.empty:

        dados = dados.drop(columns=["isPartial"], errors="ignore")
        todos_trends.append(dados)


    time.sleep(10)


df_trends = pd.concat(todos_trends, axis=1)

print(f"\n✅ Dados do Trends coletados!")
print(f"Período: {df_trends.index[0].date()} até {df_trends.index[-1].date()}")
print(f"Total de semanas: {len(df_trends)}")
df_trends.head()

🔍 Buscando trends: ['Elden Ring', "Baldur's Gate 3", 'Cyberpunk 2077', 'Stardew Valley', 'Among Us']


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



🔍 Buscando trends: ['Fall Guys', 'Hollow Knight', 'Vampire Survivors', 'Lethal Company', 'Deep Rock Galactic']


/usr/local/lib/python3.12/dist-packages/pytrends/request.py:260: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`




✅ Dados do Trends coletados!
Período: 2021-05-02 até 2026-04-26
Total de semanas: 261


,Elden Ring,Baldur's Gate 3,Cyberpunk 2077,Stardew Valley,Among Us,Fall Guys,Hollow Knight,Vampire Survivors,Lethal Company,Deep Rock Galactic
date,,,,,,,,,,
2021-05-02,0,0,1,3,6,4,4,0,0,0
2021-05-09,0,0,1,3,6,4,4,0,0,1
2021-05-16,0,0,1,3,6,4,4,0,0,1
2021-05-23,0,0,1,3,6,4,4,0,0,1
2021-05-30,0,0,1,3,6,4,4,0,0,0


In [6]:


df_trends_long = df_trends.reset_index().melt(
    id_vars="date",
    var_name="jogo",
    value_name="interesse"
)

print("✅ Tabela transformada!")
print(f"Formato: {df_trends_long.shape[0]} linhas x {df_trends_long.shape[1]} colunas")
print("\nPrimeiras linhas:")
df_trends_long.head(10)

✅ Tabela transformada!
Formato: 2610 linhas x 3 colunas

Primeiras linhas:


,date,jogo,interesse
0,2021-05-02,Elden Ring,0
1,2021-05-09,Elden Ring,0
2,2021-05-16,Elden Ring,0
3,2021-05-23,Elden Ring,0
4,2021-05-30,Elden Ring,0
5,2021-06-06,Elden Ring,2
6,2021-06-13,Elden Ring,1
7,2021-06-20,Elden Ring,0
8,2021-06-27,Elden Ring,0
9,2021-07-04,Elden Ring,0


In [7]:
fig5 = px.line(
    df_trends_long,
    x="date",
    y="interesse",
    color="jogo",
    title="📈 Ciclo de Vida — Interesse ao Longo do Tempo (Google Trends)",
    labels={
        "date"      : "Data",
        "interesse" : "Interesse (0-100)",
        "jogo"      : "Jogo"
    },
)

fig5.update_layout(
    height=550,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
    yaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
    hovermode="x unified",
)

fig5.show()

In [8]:
fig5.write_html("grafico5_trends.html")
print("✅ Salvo!")

✅ Salvo!


In [9]:
from google.colab import files
files.download("grafico5_trends.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:

pico_trends = df_trends.max().reset_index()
pico_trends.columns = ["nome", "pico_trends"]

print("🏆 Pico de interesse no Google Trends por jogo:")
print(pico_trends.sort_values("pico_trends", ascending=False).to_string(index=False))

🏆 Pico de interesse no Google Trends por jogo:
              nome  pico_trends
        Elden Ring          100
     Hollow Knight          100
         Fall Guys           93
    Lethal Company           40
   Baldur's Gate 3           20
    Stardew Valley           11
 Vampire Survivors            9
    Cyberpunk 2077            8
          Among Us            6
Deep Rock Galactic            6


In [11]:

df_cruzado = df[["nome", "total_reviews", "metacritic", "categoria"]].merge(
    pico_trends, on="nome"
)

fig6 = px.scatter(
    df_cruzado.dropna(subset=["total_reviews", "metacritic"]),
    x="pico_trends",
    y="total_reviews",
    color="categoria",
    text="nome",
    size="metacritic",
    title="🔥 Hype vs Amor Real — pico de buscas vs avaliações na Steam",
    labels={
        "pico_trends"   : "Pico de Interesse Google Trends (0-100)",
        "total_reviews" : "Total de Avaliações na Steam",
        "categoria"     : "Categoria",
        "metacritic"    : "Nota Metacritic"
    },
    color_discrete_map=cores,
    size_max=40,
)

fig6.update_traces(textposition="top center")
fig6.update_layout(
    height=550,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
    yaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
)

import numpy as np
df_temp = df_cruzado.dropna(subset=["total_reviews", "pico_trends"])
z = np.polyfit(df_temp["pico_trends"], df_temp["total_reviews"], 1)
p = np.poly1d(z)
x_line = np.linspace(df_temp["pico_trends"].min(), df_temp["pico_trends"].max(), 100)

fig6.add_trace(go.Scatter(
    x=x_line, y=p(x_line),
    mode="lines",
    line=dict(dash="dash", color="gray", width=1),
    name="tendência",
    showlegend=True,
))

fig6.show()

In [12]:
fig6.write_html("grafico6_hype_vs_amor.html")
print("✅ Salvo!")

✅ Salvo!


In [13]:
from google.colab import files
files.download("grafico6_hype_vs_amor.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>